# Tokenization — Expert

> **Section 01 — Topic 01 — expert**

**Prerequisites:** `01-tokenization/advanced.ipynb`

**What you'll learn:**
- How tokenizer choices interact with model architecture and training
- Fertility analysis and its impact on inference cost
- Tokenizer merging and extension for continual pretraining

**What you'll build:**
- A tokenizer analysis toolkit that audits any model's tokenizer

## Setup

We need `transformers` for accessing pretrained tokenizers, `tiktoken` for OpenAI's tokenizers, `matplotlib` and `seaborn` for visualization, and `numpy` for numerical work. The `sentencepiece` library gives us access to the underlying SPM models used by many open-weight LLMs.

We also import `collections` and `math` from the standard library for frequency analysis and information-theoretic calculations respectively. These will be essential for our compression ratio and fertility analyses later in the notebook.

In [ ]:
# !pip install transformers tiktoken sentencepiece matplotlib seaborn numpy

import collections
import math
import json
import re
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import tiktoken
from transformers import AutoTokenizer

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

We will load several tokenizers up front so we can compare them throughout the notebook. These represent different design philosophies: GPT-4's `cl100k_base` uses a 100K BPE vocabulary optimized for multilingual performance; LLaMA-2 uses a 32K SentencePiece BPE vocabulary that is notably smaller; and Mistral uses a 32K vocabulary with a different merge ordering. By comparing these three, we can observe how vocabulary size and merge strategy interact with downstream performance metrics.

Loading tokenizers from Hugging Face also gives us access to the full configuration, including special tokens, added tokens, and the merge table itself, which we will inspect in later sections.

In [ ]:
# Load tokenizers for comparison
enc_cl100k = tiktoken.get_encoding("cl100k_base")      # GPT-4
enc_o200k = tiktoken.get_encoding("o200k_base")         # GPT-4o
tok_llama2 = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tok_mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

TOKENIZERS = {
    "cl100k (GPT-4)": lambda text: enc_cl100k.encode(text),
    "o200k (GPT-4o)": lambda text: enc_o200k.encode(text),
    "LLaMA-2-7B": lambda text: tok_llama2.encode(text, add_special_tokens=False),
    "Mistral-7B": lambda text: tok_mistral.encode(text, add_special_tokens=False),
}

VOCAB_SIZES = {
    "cl100k (GPT-4)": enc_cl100k.n_vocab,
    "o200k (GPT-4o)": enc_o200k.n_vocab,
    "LLaMA-2-7B": tok_llama2.vocab_size,
    "Mistral-7B": tok_mistral.vocab_size,
}

print("Loaded tokenizers with vocab sizes:")
for name, size in VOCAB_SIZES.items():
    print(f"  {name}: {size:,}")

---

## 1. Tokenizer-Model Interactions

The tokenizer is not merely a preprocessing step — it is deeply coupled to the model's architecture. The vocabulary size `V` directly determines the shape of the embedding matrix `E` (dimensions `V x d_model`) and the output projection matrix (dimensions `d_model x V`). For a model with `d_model = 4096`, moving from a 32K vocabulary to a 100K vocabulary adds roughly `(100000 - 32000) * 4096 * 2 = 557 million` parameters across input and output embeddings alone. This is not a trivial cost — it is comparable to several transformer layers.

Beyond raw parameter count, vocabulary size affects the softmax computation at every generation step. The model must compute logits over all `V` tokens, apply softmax, and sample. Larger vocabularies mean more computation in the final layer, though this is often masked by the cost of attention in deeper models. The tradeoff is that a larger vocabulary produces fewer tokens per input, meaning fewer forward passes during autoregressive generation.

The interaction between tokenization boundaries and learned representations is subtle but important. When a word is split into multiple tokens, the model must learn to compose the meaning across those tokens using self-attention. Single-token words can be looked up directly from the embedding table. This means that common words in the training data benefit from single-token representations, while rare words pay a compositional tax. This has real consequences for languages that are underrepresented in the training data.

In [ ]:
def compute_embedding_params(vocab_size: int, d_model: int, tie_weights: bool = True) -> dict:
    """Calculate embedding parameter count for a given vocab size and model dim."""
    input_params = vocab_size * d_model
    output_params = 0 if tie_weights else vocab_size * d_model
    total = input_params + output_params
    return {
        "vocab_size": vocab_size,
        "d_model": d_model,
        "input_embedding_params": input_params,
        "output_projection_params": output_params,
        "total_embedding_params": total,
        "memory_fp16_gb": total * 2 / (1024**3),
        "memory_fp32_gb": total * 4 / (1024**3),
    }

# Compare embedding costs across tokenizers
d_model = 4096  # typical for 7B models
print(f"Embedding parameter analysis (d_model={d_model}, tied weights):\n")
print(f"{'Tokenizer':<20} {'Vocab':>8} {'Params (M)':>12} {'FP16 (MB)':>10} {'FP32 (MB)':>10}")
print("-" * 64)
for name, vsize in VOCAB_SIZES.items():
    info = compute_embedding_params(vsize, d_model, tie_weights=True)
    print(f"{name:<20} {vsize:>8,} {info['total_embedding_params']/1e6:>12.1f} "
          f"{info['memory_fp16_gb']*1024:>10.1f} {info['memory_fp32_gb']*1024:>10.1f}")

In [ ]:
def estimate_flops_per_token(d_model: int, n_layers: int, n_heads: int, seq_len: int) -> float:
    """Rough FLOPs estimate per token for a transformer forward pass.
    
    Uses the approximation from Kaplan et al. and Hoffmann et al.:
    ~6 * N parameters for forward + backward, ~2 * N for forward only.
    Here we break it down by component for more insight.
    """
    d_head = d_model // n_heads
    # Per layer: QKV projection + attention + output projection + FFN
    qkv_flops = 3 * 2 * d_model * d_model  # Q, K, V projections
    attn_flops = 2 * seq_len * d_model       # attention scores + weighted sum
    out_proj_flops = 2 * d_model * d_model    # output projection
    ffn_flops = 2 * 2 * d_model * (4 * d_model)  # two linear layers with 4x expansion
    per_layer = qkv_flops + attn_flops + out_proj_flops + ffn_flops
    return per_layer * n_layers


def compare_tokenizer_flops(text: str, d_model: int = 4096, n_layers: int = 32,
                             n_heads: int = 32, seq_len: int = 2048) -> dict:
    """Compare total FLOPs to process the same text across tokenizers."""
    flops_per_tok = estimate_flops_per_token(d_model, n_layers, n_heads, seq_len)
    results = {}
    for name, encode_fn in TOKENIZERS.items():
        n_tokens = len(encode_fn(text))
        total_flops = n_tokens * flops_per_tok
        results[name] = {"n_tokens": n_tokens, "total_flops": total_flops}
    return results


# Sample text for comparison
sample_text = """The transformer architecture, introduced in 'Attention Is All You Need' (2017),
revolutionized natural language processing. Large language models like GPT-4, LLaMA, and
Mistral build on this foundation, each making different tradeoffs in tokenization strategy,
model size, and training data composition. Understanding how tokenization interacts with
the model architecture is essential for practitioners who want to optimize inference cost,
extend models to new domains, or build efficient serving pipelines."""

results = compare_tokenizer_flops(sample_text)

print(f"Text length: {len(sample_text)} characters\n")
print(f"{'Tokenizer':<20} {'Tokens':>8} {'Total GFLOPs':>14} {'Relative':>10}")
print("-" * 56)
min_flops = min(r["total_flops"] for r in results.values())
for name, r in results.items():
    print(f"{name:<20} {r['n_tokens']:>8} {r['total_flops']/1e9:>14.2f} {r['total_flops']/min_flops:>10.2f}x")

In [ ]:
# Visualize how different tokenizers split the same word
def show_tokenization_boundaries(word: str):
    """Show how each tokenizer splits a single word."""
    print(f"Word: '{word}'\n")
    for name, encode_fn in TOKENIZERS.items():
        token_ids = encode_fn(word)
        # Decode each token individually to see the pieces
        if "cl100k" in name:
            pieces = [enc_cl100k.decode([tid]) for tid in token_ids]
        elif "o200k" in name:
            pieces = [enc_o200k.decode([tid]) for tid in token_ids]
        elif "LLaMA" in name:
            pieces = [tok_llama2.decode([tid]) for tid in token_ids]
        else:
            pieces = [tok_mistral.decode([tid]) for tid in token_ids]
        print(f"  {name:<20} [{len(token_ids)} tokens]: {pieces}")
    print()


test_words = [
    "tokenization",
    "transformerarchitektur",  # German compound word
    "pneumonoultramicroscopicsilicovolcanoconiosis",
    "日本語のテキスト",  # Japanese text
    "Ñoño",  # Spanish with special chars
    "defenestration",
]

for word in test_words:
    show_tokenization_boundaries(word)

The results above demonstrate a key insight: tokenizer efficiency is not uniform across languages or word types. English common words tend to be single tokens across all tokenizers, while rare English words and non-Latin scripts show dramatic differences. The larger vocabularies (100K+) tend to compress multilingual text more efficiently, but this comes at the cost of a larger embedding matrix.

Notice how compound words and technical terms are split differently. The model must learn to compose meaning from these fragments, which means the attention mechanism carries extra burden. For languages where the tokenizer produces many tokens per word (high fertility), the model effectively has less context window capacity per word. A 4096-token context window might hold 3000 English words but only 1500 words of an underrepresented language.

---

## 2. Fertility Analysis Deep Dive

Fertility is defined as the average number of tokens produced per word (or per character, depending on the convention). It is the single most important metric for understanding tokenizer efficiency in a given language or domain. A fertility of 1.0 means every word maps to exactly one token — the ideal case. A fertility of 3.0 means the average word is split into three tokens, tripling the sequence length and inference cost.

Fertility varies dramatically across languages. For BPE tokenizers trained predominantly on English web text, English fertility is typically 1.1-1.3, while morphologically rich languages like Finnish or Turkish may see fertility of 2.0-3.0, and logographic languages like Chinese or Japanese can reach even higher values. This disparity has real economic consequences: API pricing is per-token, so the same semantic content costs 2-3x more to process in an underrepresented language.

Beyond language, fertility also varies by domain. Legal text, medical terminology, and source code all have distinctive vocabulary distributions. A tokenizer trained on web text will have poor fertility on specialized domains, which is one motivation for domain-adaptive tokenizer extension (covered in Section 4).

In [ ]:
class FertilityAnalyzer:
    """Analyze tokenizer fertility across languages and domains."""
    
    def __init__(self, tokenizers: Dict[str, callable]):
        self.tokenizers = tokenizers
    
    def word_fertility(self, text: str, tokenizer_name: str) -> float:
        """Average tokens per whitespace-separated word."""
        words = text.split()
        if not words:
            return 0.0
        encode_fn = self.tokenizers[tokenizer_name]
        total_tokens = len(encode_fn(text))
        return total_tokens / len(words)
    
    def char_fertility(self, text: str, tokenizer_name: str) -> float:
        """Average tokens per character (inverse of compression ratio)."""
        if not text:
            return 0.0
        encode_fn = self.tokenizers[tokenizer_name]
        total_tokens = len(encode_fn(text))
        return total_tokens / len(text)
    
    def analyze_corpus(self, corpus: Dict[str, str]) -> dict:
        """Compute fertility for each (tokenizer, corpus_entry) pair."""
        results = {}
        for corpus_name, text in corpus.items():
            results[corpus_name] = {}
            for tok_name in self.tokenizers:
                results[corpus_name][tok_name] = {
                    "word_fertility": self.word_fertility(text, tok_name),
                    "char_fertility": self.char_fertility(text, tok_name),
                    "n_tokens": len(self.tokenizers[tok_name](text)),
                    "n_words": len(text.split()),
                    "n_chars": len(text),
                }
        return results


analyzer = FertilityAnalyzer(TOKENIZERS)
print("FertilityAnalyzer ready.")

In [ ]:
# Multilingual + multi-domain corpus for fertility analysis
corpus = {
    "English (news)": (
        "The Federal Reserve announced a quarter-point interest rate cut on Wednesday, "
        "signaling confidence that inflation is moving sustainably toward its two percent "
        "target. Markets rallied on the news, with the S&P 500 closing at a record high. "
        "Economists noted that the labor market remains resilient despite slowing growth "
        "in the manufacturing sector."
    ),
    "English (code)": (
        "def transformer_block(x, attn_mask=None, kv_cache=None):\n"
        "    residual = x\n"
        "    x = self.layer_norm1(x)\n"
        "    x, new_kv = self.self_attention(x, attn_mask, kv_cache)\n"
        "    x = residual + x\n"
        "    residual = x\n"
        "    x = self.layer_norm2(x)\n"
        "    x = self.feed_forward(x)\n"
        "    return residual + x, new_kv\n"
    ),
    "English (medical)": (
        "The patient presented with acute cholecystitis complicated by choledocholithiasis. "
        "Magnetic resonance cholangiopancreatography revealed a 12mm calculus in the "
        "common bile duct with upstream biliary dilatation. Endoscopic retrograde "
        "cholangiopancreatography with sphincterotomy was performed successfully."
    ),
    "French": (
        "L'apprentissage automatique a considérablement transformé le traitement du langage "
        "naturel au cours de la dernière décennie. Les modèles de grande taille, entraînés "
        "sur des corpus massifs de texte, démontrent des capacités remarquables en matière "
        "de compréhension et de génération de texte."
    ),
    "German": (
        "Die Künstliche Intelligenz hat in den letzten Jahren bemerkenswerte Fortschritte "
        "erzielt. Besonders die Verarbeitung natürlicher Sprache hat durch große "
        "Sprachmodelle einen Quantensprung erlebt. Zusammengesetzte Wörter wie "
        "Geschwindigkeitsbeschränkung stellen besondere Herausforderungen dar."
    ),
    "Chinese": (
        "大型语言模型在自然语言处理领域取得了显著的进展。这些模型通过在大规模文本数据上进行预训练，"
        "能够理解和生成高质量的文本内容。研究人员正在探索如何使这些模型更加高效和可靠。"
    ),
    "Japanese": (
        "大規模言語モデルは自然言語処理の分野で顕著な進歩を遂げています。"
        "これらのモデルは大量のテキストデータで事前学習され、高品質なテキストの理解と生成が可能です。"
        "研究者たちはこれらのモデルをより効率的で信頼性の高いものにする方法を模索しています。"
    ),
    "Arabic": (
        "حققت نماذج اللغة الكبيرة تقدما ملحوظا في مجال معالجة اللغة الطبيعية. "
        "يتم تدريب هذه النماذج على كميات هائلة من البيانات النصية لتتمكن من فهم "
        "وتوليد نصوص عالية الجودة."
    ),
}

results = analyzer.analyze_corpus(corpus)

# Display fertility table
print(f"{'Corpus':<22}", end="")
for tok_name in TOKENIZERS:
    print(f"{tok_name:>18}", end="")
print()
print("-" * 94)
for corpus_name in corpus:
    print(f"{corpus_name:<22}", end="")
    for tok_name in TOKENIZERS:
        fert = results[corpus_name][tok_name]["word_fertility"]
        print(f"{fert:>18.2f}", end="")
    print()

In [ ]:
# Fertility heatmap
corpus_names = list(corpus.keys())
tok_names = list(TOKENIZERS.keys())

fertility_matrix = np.array([
    [results[c][t]["word_fertility"] for t in tok_names]
    for c in corpus_names
])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(fertility_matrix, cmap="YlOrRd", aspect="auto")

ax.set_xticks(range(len(tok_names)))
ax.set_xticklabels(tok_names, rotation=30, ha="right")
ax.set_yticks(range(len(corpus_names)))
ax.set_yticklabels(corpus_names)

# Annotate each cell
for i in range(len(corpus_names)):
    for j in range(len(tok_names)):
        val = fertility_matrix[i, j]
        color = "white" if val > fertility_matrix.mean() + 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=9)

plt.colorbar(im, ax=ax, label="Word Fertility (tokens/word)")
ax.set_title("Tokenizer Fertility by Language and Domain")
plt.tight_layout()
plt.show()

In [ ]:
def estimate_api_cost(text: str, tokenizer_name: str, cost_per_1k_input: float = 0.01,
                       cost_per_1k_output: float = 0.03) -> dict:
    """Estimate API cost for processing text with a given tokenizer.
    
    Default prices approximate GPT-4-turbo pricing as of early 2025.
    """
    encode_fn = TOKENIZERS[tokenizer_name]
    n_tokens = len(encode_fn(text))
    input_cost = (n_tokens / 1000) * cost_per_1k_input
    # Assume output is ~50% of input length for estimation
    output_cost = (n_tokens * 0.5 / 1000) * cost_per_1k_output
    return {
        "n_tokens": n_tokens,
        "input_cost_usd": input_cost,
        "output_cost_usd": output_cost,
        "total_cost_usd": input_cost + output_cost,
    }


# Compare costs across languages for a fixed semantic content length
print("Estimated API cost for equivalent-length texts (per request):\n")
print(f"{'Language/Domain':<22} {'Tokens':>8} {'Input ($)':>10} {'Total ($)':>10} {'Relative':>10}")
print("-" * 64)

baseline_cost = None
for corpus_name, text in corpus.items():
    # Use cl100k as representative
    cost_info = estimate_api_cost(text, "cl100k (GPT-4)")
    if baseline_cost is None:
        baseline_cost = cost_info["total_cost_usd"]
    relative = cost_info["total_cost_usd"] / baseline_cost if baseline_cost > 0 else 0
    print(f"{corpus_name:<22} {cost_info['n_tokens']:>8} "
          f"{cost_info['input_cost_usd']:>10.6f} {cost_info['total_cost_usd']:>10.6f} "
          f"{relative:>10.2f}x")

The heatmap above reveals the core inequality of modern tokenizers. Languages well-represented in the training data (English, French) enjoy low fertility and thus lower inference costs. Languages with different scripts (Chinese, Japanese, Arabic) or rich morphology (German) pay a significant premium. This is not merely an academic concern — it directly affects who can afford to use these models and how effective they are.

The economic argument is straightforward: if Chinese text produces 2x the tokens of English text for the same semantic content, then Chinese users pay 2x the API cost. At scale, this amounts to millions of dollars in differential pricing for the same capability. This has motivated work on multilingual tokenizer design and tokenizer extension for underserved languages.

---

## 3. Compression Ratios and Information Theory

A tokenizer is fundamentally a compression algorithm. BPE, WordPiece, and Unigram all learn to represent frequent byte sequences as single tokens, reducing the total number of symbols needed to encode a text. We can measure this compression using information-theoretic metrics, which let us compare tokenizers to theoretically optimal compression.

The key metric is **bits per character** (BPC). If a tokenizer uses a vocabulary of size `V` and produces `T` tokens for a text of `C` characters, then the naive BPC is `T * log2(V) / C`. This tells us how many bits of information the tokenizer uses per original character. A lower BPC means better compression. The theoretical minimum is the entropy of the character distribution in the text.

In practice, the model's cross-entropy loss provides a tighter bound on the effective BPC, since the model learns to assign higher probability to likely next tokens. But the tokenizer's compression ratio sets the ceiling — a poor tokenizer forces the model to waste capacity encoding redundant structure rather than learning semantics.

In [ ]:
def char_entropy(text: str) -> float:
    """Compute character-level entropy of text in bits."""
    freq = collections.Counter(text)
    total = len(text)
    entropy = 0.0
    for count in freq.values():
        p = count / total
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy


def tokenizer_bpc(text: str, tokenizer_name: str) -> float:
    """Bits per character using a given tokenizer.
    
    BPC = (n_tokens * log2(vocab_size)) / n_characters
    This is the naive upper bound — the model's learned distribution will be tighter.
    """
    encode_fn = TOKENIZERS[tokenizer_name]
    n_tokens = len(encode_fn(text))
    n_chars = len(text)
    vocab_size = VOCAB_SIZES[tokenizer_name]
    return (n_tokens * math.log2(vocab_size)) / n_chars


def compression_ratio(text: str, tokenizer_name: str) -> float:
    """Bytes in original text / bytes to encode as token IDs.
    
    Token IDs are stored as integers, so we compute the number of bytes needed
    to represent the token sequence using ceil(log2(vocab_size)) bits per token.
    """
    encode_fn = TOKENIZERS[tokenizer_name]
    n_tokens = len(encode_fn(text))
    vocab_size = VOCAB_SIZES[tokenizer_name]
    bits_per_token = math.ceil(math.log2(vocab_size))
    encoded_bytes = (n_tokens * bits_per_token) / 8
    original_bytes = len(text.encode("utf-8"))
    return original_bytes / encoded_bytes if encoded_bytes > 0 else 0


print("Compression analysis on English news corpus:\n")
text = corpus["English (news)"]
h_char = char_entropy(text)
print(f"Character entropy: {h_char:.3f} bits/char")
print(f"UTF-8 encoding:    {8.0:.3f} bits/char (naive)")
print()

print(f"{'Tokenizer':<20} {'BPC':>8} {'Compression':>12} {'Tokens':>8}")
print("-" * 52)
for tok_name in TOKENIZERS:
    bpc = tokenizer_bpc(text, tok_name)
    cr = compression_ratio(text, tok_name)
    n_tok = len(TOKENIZERS[tok_name](text))
    print(f"{tok_name:<20} {bpc:>8.3f} {cr:>12.2f}x {n_tok:>8}")

In [ ]:
# Compare BPC across all corpus entries and tokenizers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BPC comparison
bpc_data = {}
for corpus_name, text in corpus.items():
    bpc_data[corpus_name] = {}
    for tok_name in TOKENIZERS:
        bpc_data[corpus_name][tok_name] = tokenizer_bpc(text, tok_name)

x = np.arange(len(corpus))
width = 0.2
for i, tok_name in enumerate(TOKENIZERS):
    values = [bpc_data[c][tok_name] for c in corpus]
    axes[0].bar(x + i * width, values, width, label=tok_name)

axes[0].set_xlabel("Corpus")
axes[0].set_ylabel("Bits per Character (BPC)")
axes[0].set_title("Naive BPC by Tokenizer and Corpus")
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(list(corpus.keys()), rotation=45, ha="right", fontsize=7)
axes[0].legend(fontsize=7)

# Compression ratio comparison
cr_data = {}
for corpus_name, text in corpus.items():
    cr_data[corpus_name] = {}
    for tok_name in TOKENIZERS:
        cr_data[corpus_name][tok_name] = compression_ratio(text, tok_name)

for i, tok_name in enumerate(TOKENIZERS):
    values = [cr_data[c][tok_name] for c in corpus]
    axes[1].bar(x + i * width, values, width, label=tok_name)

axes[1].set_xlabel("Corpus")
axes[1].set_ylabel("Compression Ratio (original / encoded)")
axes[1].set_title("Compression Ratio by Tokenizer and Corpus")
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(list(corpus.keys()), rotation=45, ha="right", fontsize=7)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

These results illustrate a fundamental tension in tokenizer design. Larger vocabularies achieve better compression on well-represented text but carry higher memory cost. The naive BPC is always far above the character entropy, reminding us that tokenizers are not optimal compressors — they are constrained by the BPE greedy merge heuristic and the need for a fixed vocabulary.

The gap between tokenizer BPC and character entropy represents the opportunity cost of the tokenization scheme. More sophisticated tokenizer training (e.g., Unigram with proper EM optimization) can narrow this gap, but the real compression happens inside the model itself, where the learned probability distribution over next tokens provides the actual coding scheme.

---

## 4. Tokenizer Extension and Merging

When adapting a pretrained model to a new domain or language, modifying the tokenizer is often necessary. If the target domain uses vocabulary that the original tokenizer splits into many subword pieces (high fertility), the model wastes context window capacity and inference FLOPs on token-level composition instead of semantic reasoning. Adding domain-specific tokens can dramatically improve fertility and thus model performance.

However, extending a tokenizer is not as simple as appending new entries to the vocabulary. Each new token requires a corresponding embedding vector in the model, and these embeddings must be initialized carefully. Random initialization works but requires significant fine-tuning. Mean initialization (averaging embeddings of the token's constituent sub-tokens) provides a better starting point. Semantic initialization (using a separate embedding model) is more sophisticated but can accelerate convergence.

Merging two tokenizers — combining vocabularies from models trained on different data — is even more complex. Overlapping tokens must be reconciled, merge priorities must be resolved, and the resulting vocabulary must remain compatible with at least one of the original models. This technique is used in multilingual model adaptation and in building mixture-of-experts systems that share a tokenizer.

In [ ]:
class TokenizerExtender:
    """Extend a HuggingFace tokenizer with new domain-specific tokens."""
    
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.original_vocab_size = len(tokenizer)
        self.added_tokens = []
    
    def find_high_fertility_terms(self, domain_texts: List[str], 
                                   min_fertility: float = 2.0,
                                   min_freq: int = 3) -> List[Tuple[str, float, int]]:
        """Identify domain terms that would benefit from dedicated tokens."""
        word_freq = collections.Counter()
        word_fertility = {}
        
        for text in domain_texts:
            words = text.split()
            word_freq.update(words)
        
        for word, freq in word_freq.items():
            if freq >= min_freq:
                tokens = self.tokenizer.encode(word, add_special_tokens=False)
                fertility = len(tokens)
                if fertility >= min_fertility:
                    word_fertility[word] = (fertility, freq)
        
        # Sort by (fertility * frequency) — impact score
        ranked = sorted(
            [(w, f, c) for w, (f, c) in word_fertility.items()],
            key=lambda x: x[1] * x[2],
            reverse=True
        )
        return ranked
    
    def add_tokens(self, new_tokens: List[str]) -> int:
        """Add new tokens to the tokenizer."""
        num_added = self.tokenizer.add_tokens(new_tokens)
        self.added_tokens.extend(new_tokens)
        return num_added
    
    def initialize_embeddings_mean(self, model_embeddings, new_token_strings: List[str]):
        """Initialize new token embeddings as mean of sub-token embeddings.
        
        This simulates what you'd do with a real model — we work with numpy here.
        """
        d_model = model_embeddings.shape[1]
        new_embeddings = []
        
        for token_str in new_token_strings:
            # Encode the string using original tokenizer to get sub-tokens
            sub_token_ids = self.tokenizer.encode(token_str, add_special_tokens=False)
            # Filter to valid original IDs
            valid_ids = [tid for tid in sub_token_ids if tid < self.original_vocab_size]
            if valid_ids:
                mean_emb = model_embeddings[valid_ids].mean(axis=0)
            else:
                mean_emb = np.random.randn(d_model) * 0.02
            new_embeddings.append(mean_emb)
        
        return np.array(new_embeddings)
    
    def report(self):
        """Print extension summary."""
        print(f"Original vocab size: {self.original_vocab_size:,}")
        print(f"Added tokens: {len(self.added_tokens)}")
        print(f"New vocab size: {len(self.tokenizer):,}")
        return self.added_tokens


print("TokenizerExtender class defined.")

In [ ]:
# Demonstrate tokenizer extension with medical domain text
medical_texts = [
    "The patient presented with cholecystitis and choledocholithiasis requiring cholecystectomy.",
    "Magnetic resonance cholangiopancreatography confirmed choledocholithiasis with biliary dilatation.",
    "Endoscopic retrograde cholangiopancreatography with sphincterotomy was performed.",
    "Post-cholecystectomy syndrome may include choledocholithiasis recurrence.",
    "The cholangiopancreatography revealed cholecystitis with pericholecystic fluid.",
    "Laparoscopic cholecystectomy is the standard treatment for acute cholecystitis.",
    "Cholangiopancreatography is essential for evaluating choledocholithiasis.",
    "The sphincterotomy successfully resolved the choledocholithiasis.",
]

# Use a fresh copy of the Mistral tokenizer
tok_fresh = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
extender = TokenizerExtender(tok_fresh)

# Find high-fertility medical terms
candidates = extender.find_high_fertility_terms(medical_texts, min_fertility=2.0, min_freq=2)

print("High-fertility medical terms (candidates for new tokens):\n")
print(f"{'Term':<40} {'Fertility':>10} {'Frequency':>10} {'Impact':>10}")
print("-" * 74)
for term, fertility, freq in candidates[:15]:
    print(f"{term:<40} {fertility:>10} {freq:>10} {fertility * freq:>10}")

In [ ]:
# Add top candidates as new tokens and measure fertility improvement
top_candidates = [term for term, _, _ in candidates[:8]]

# Measure before
test_text = " ".join(medical_texts)
tokens_before = tok_fresh.encode(test_text, add_special_tokens=False)
print(f"Before extension: {len(tokens_before)} tokens for {len(test_text)} chars")
print(f"  Fertility: {len(tokens_before) / len(test_text.split()):.2f} tokens/word")

# Add tokens
num_added = extender.add_tokens(top_candidates)
print(f"\nAdded {num_added} new tokens: {top_candidates}")

# Measure after
tokens_after = tok_fresh.encode(test_text, add_special_tokens=False)
print(f"\nAfter extension: {len(tokens_after)} tokens for {len(test_text)} chars")
print(f"  Fertility: {len(tokens_after) / len(test_text.split()):.2f} tokens/word")
print(f"  Token reduction: {len(tokens_before) - len(tokens_after)} tokens "
      f"({(1 - len(tokens_after)/len(tokens_before))*100:.1f}%)")

extender.report()

In [ ]:
# Simulate embedding initialization strategies
d_model = 256  # small for demonstration
original_vocab_size = extender.original_vocab_size

# Simulate original embeddings (normally these come from the model)
np.random.seed(42)
fake_embeddings = np.random.randn(original_vocab_size, d_model).astype(np.float32) * 0.02

# Strategy 1: Random initialization
random_init = np.random.randn(len(top_candidates), d_model).astype(np.float32) * 0.02

# Strategy 2: Mean of sub-token embeddings
mean_init = extender.initialize_embeddings_mean(fake_embeddings, top_candidates)

# Strategy 3: Zero initialization (baseline)
zero_init = np.zeros((len(top_candidates), d_model), dtype=np.float32)

# Compare the initialization strategies
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

strategies = [
    ("Random Init", random_init),
    ("Mean Sub-token Init", mean_init),
    ("Zero Init", zero_init),
]

for ax, (name, embeddings) in zip(axes, strategies):
    # Show embedding norms
    norms = np.linalg.norm(embeddings, axis=1)
    ax.bar(range(len(top_candidates)), norms, color="steelblue")
    ax.set_title(name)
    ax.set_xlabel("New Token Index")
    ax.set_ylabel("L2 Norm")
    ax.set_ylim(0, max(1.0, norms.max() * 1.2) if norms.max() > 0 else 1.0)

plt.suptitle("Embedding Initialization Strategies for New Tokens", y=1.02)
plt.tight_layout()
plt.show()

print("\nNorm statistics:")
for name, embeddings in strategies:
    norms = np.linalg.norm(embeddings, axis=1)
    print(f"  {name:<25} mean={norms.mean():.4f}  std={norms.std():.4f}")

# Compare to original embedding norms
orig_norms = np.linalg.norm(fake_embeddings[:1000], axis=1)
print(f"  {'Original embeddings':<25} mean={orig_norms.mean():.4f}  std={orig_norms.std():.4f}")

In [ ]:
def merge_tokenizer_vocabs(tok_a, tok_b, max_new_tokens: int = 1000,
                            min_frequency_rank: int = 5000) -> List[str]:
    """Find tokens in tok_b's vocabulary that are missing from tok_a.
    
    Returns a list of candidate tokens to add to tok_a, prioritized by
    their rank in tok_b's vocabulary (lower ID = higher priority, as BPE
    assigns lower IDs to more frequent merges).
    """
    vocab_a = set(tok_a.get_vocab().keys())
    vocab_b = tok_b.get_vocab()  # {token_str: token_id}
    
    # Find tokens in B but not in A, sorted by ID in B (proxy for frequency)
    unique_to_b = [
        (token, token_id) 
        for token, token_id in vocab_b.items()
        if token not in vocab_a and token_id < min_frequency_rank
    ]
    unique_to_b.sort(key=lambda x: x[1])  # sort by token_id (frequency proxy)
    
    candidates = [token for token, _ in unique_to_b[:max_new_tokens]]
    return candidates


# Demonstrate vocab merging between LLaMA-2 and Mistral
tok_a = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tok_b = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

merge_candidates = merge_tokenizer_vocabs(tok_a, tok_b, max_new_tokens=20)

print(f"Vocab overlap analysis:")
print(f"  LLaMA-2 vocab size: {len(tok_a.get_vocab()):,}")
print(f"  Mistral vocab size: {len(tok_b.get_vocab()):,}")

vocab_a_set = set(tok_a.get_vocab().keys())
vocab_b_set = set(tok_b.get_vocab().keys())
overlap = vocab_a_set & vocab_b_set
print(f"  Overlap: {len(overlap):,} tokens")
print(f"  Unique to LLaMA-2: {len(vocab_a_set - vocab_b_set):,}")
print(f"  Unique to Mistral: {len(vocab_b_set - vocab_a_set):,}")
print(f"\nTop merge candidates (Mistral tokens missing from LLaMA-2):")
for i, token in enumerate(merge_candidates[:20]):
    print(f"  {i+1:>3}. {repr(token)}")

### Pitfalls of Tokenizer Extension

Extending a tokenizer is powerful but dangerous if done carelessly. The most common pitfall is **breaking existing tokenization patterns**. When you add a new token that is a substring of an existing multi-token sequence, the tokenizer may now produce different output for inputs that previously worked correctly. This means the model's learned representations for those inputs become invalid.

For example, if you add "cholecyst" as a new token, the word "cholecystectomy" might now tokenize as ["cholecyst", "ectomy"] instead of whatever fragments the original tokenizer used. Any fine-tuned behavior that depended on the old tokenization is now broken. The model must re-learn the association between the new token sequence and the target output.

A second pitfall is **vocabulary bloat**. Each new token adds parameters to the embedding and output projection matrices. Adding 10,000 tokens to a model with `d_model=4096` adds ~82M parameters (41M for input embeddings + 41M for output projection if untied). This is not free — it increases memory usage and slightly slows down the softmax computation at every generation step.

---

## 5. Token Healing

Token healing addresses a subtle but impactful problem in language model generation. When a prompt is tokenized and the model generates a continuation, the boundary between prompt and generation can create **tokenization artifacts**. This happens because the tokenizer processes the full concatenated string differently than the two pieces separately.

Consider a prompt that ends with `"The answer is "` and a desired continuation of `"42"`. If we tokenize the prompt alone, the last token might be `" is"` with a trailing space. But when the model generates `"42"`, the tokenizer for the full string `"The answer is 42"` might merge the space with `"42"` into a single token `" 42"`. The model never sees this merged token during generation because it is forced to continue from the prompt's last token boundary.

This discrepancy means the model's output probability distribution is conditioned on an artificial token boundary that would not exist if the full text were tokenized at once. Token healing resolves this by backing up one or more tokens from the prompt boundary and re-tokenizing. This is especially critical for **constrained generation** (JSON, code, structured output), where exact character sequences matter and token boundary artifacts can cause parse failures.

In [ ]:
def demonstrate_token_boundary_problem(prompt: str, continuation: str, enc):
    """Show how tokenization differs at the prompt/generation boundary."""
    # Tokenize separately
    prompt_tokens = enc.encode(prompt)
    continuation_tokens = enc.encode(continuation)
    separate_tokens = prompt_tokens + continuation_tokens
    
    # Tokenize together
    full_text = prompt + continuation
    joint_tokens = enc.encode(full_text)
    
    # Decode each token for visualization
    sep_decoded = [enc.decode([t]) for t in separate_tokens]
    joint_decoded = [enc.decode([t]) for t in joint_tokens]
    
    print(f"Prompt:       {repr(prompt)}")
    print(f"Continuation: {repr(continuation)}")
    print(f"Full text:    {repr(full_text)}")
    print()
    print(f"Separate tokenization ({len(separate_tokens)} tokens):")
    print(f"  Prompt tokens:  {sep_decoded[:len(prompt_tokens)]}")
    print(f"  Cont. tokens:   {sep_decoded[len(prompt_tokens):]}")
    print(f"Joint tokenization ({len(joint_tokens)} tokens):")
    print(f"  All tokens:     {joint_decoded}")
    
    mismatch = separate_tokens != joint_tokens
    print(f"\nMismatch: {mismatch}")
    if mismatch:
        print(f"  Token count difference: {len(separate_tokens) - len(joint_tokens)}")
    print()
    return mismatch


# Examples that commonly trigger boundary artifacts
examples = [
    ("The answer is ", "42"),
    ('{ "name": "', 'Alice" }'),
    ("def compute_", "loss(x, y):"),
    ("https://", "example.com/api"),
    ("The value is $", "100.00"),
]

print("=" * 70)
print("TOKEN BOUNDARY PROBLEM DEMONSTRATION (cl100k_base)")
print("=" * 70)
for prompt, cont in examples:
    demonstrate_token_boundary_problem(prompt, cont, enc_cl100k)
    print("-" * 70)

In [ ]:
class TokenHealer:
    """Implement token healing for generation boundaries.
    
    Token healing works by:
    1. Taking the last N tokens of the prompt
    2. Decoding them back to text
    3. Re-tokenizing the decoded text + generated text together
    4. Returning the correctly tokenized continuation
    """
    
    def __init__(self, encoding):
        self.enc = encoding
    
    def heal(self, prompt_tokens: List[int], generated_text: str,
             n_backup: int = 1) -> Tuple[List[int], List[int]]:
        """Heal the token boundary between prompt and generation.
        
        Args:
            prompt_tokens: Token IDs for the prompt
            generated_text: Raw text generated by the model
            n_backup: Number of prompt tokens to back up for re-tokenization
        
        Returns:
            (healed_prompt_tokens, healed_continuation_tokens)
        """
        if n_backup <= 0 or not prompt_tokens:
            return prompt_tokens, self.enc.encode(generated_text)
        
        # Back up n tokens from the prompt
        backup_tokens = prompt_tokens[-n_backup:]
        stable_tokens = prompt_tokens[:-n_backup]
        
        # Decode the backup region and concatenate with generated text
        backup_text = self.enc.decode(backup_tokens)
        combined_text = backup_text + generated_text
        
        # Re-tokenize the combined region
        healed_tokens = self.enc.encode(combined_text)
        
        # The full sequence is stable_tokens + healed_tokens
        healed_prompt_len = len(stable_tokens)
        
        return stable_tokens, healed_tokens
    
    def compare(self, prompt: str, continuation: str, n_backup: int = 1):
        """Compare naive vs healed tokenization."""
        prompt_tokens = self.enc.encode(prompt)
        
        # Naive: just concatenate token lists
        naive_cont_tokens = self.enc.encode(continuation)
        naive_full = prompt_tokens + naive_cont_tokens
        
        # Healed
        stable, healed_region = self.heal(prompt_tokens, continuation, n_backup)
        healed_full = list(stable) + list(healed_region)
        
        # Ground truth: tokenize the full string
        truth = self.enc.encode(prompt + continuation)
        
        return {
            "naive": naive_full,
            "healed": healed_full,
            "ground_truth": truth,
            "naive_matches_truth": naive_full == truth,
            "healed_matches_truth": healed_full == truth,
        }


healer = TokenHealer(enc_cl100k)

print("Token Healing Results\n")
for prompt, cont in examples:
    result = healer.compare(prompt, cont, n_backup=1)
    print(f"Prompt: {repr(prompt)} + Cont: {repr(cont)}")
    print(f"  Naive matches ground truth:  {result['naive_matches_truth']}")
    print(f"  Healed matches ground truth: {result['healed_matches_truth']}")
    if not result['healed_matches_truth']:
        # Try with more backup
        result2 = healer.compare(prompt, cont, n_backup=2)
        print(f"  Healed (n=2) matches truth: {result2['healed_matches_truth']}")
    print()

In [ ]:
# Visualize where token boundary mismatches occur
def scan_for_boundary_artifacts(prompts: List[str], continuations: List[str], enc) -> dict:
    """Scan many prompt/continuation pairs for tokenization artifacts."""
    results = {"mismatch": 0, "match": 0, "total_token_diff": 0}
    mismatches = []
    
    for prompt, cont in zip(prompts, continuations):
        sep = enc.encode(prompt) + enc.encode(cont)
        joint = enc.encode(prompt + cont)
        if sep != joint:
            results["mismatch"] += 1
            results["total_token_diff"] += abs(len(sep) - len(joint))
            mismatches.append((prompt, cont, len(sep) - len(joint)))
        else:
            results["match"] += 1
    
    results["mismatch_rate"] = results["mismatch"] / max(1, results["mismatch"] + results["match"])
    results["examples"] = mismatches
    return results


# Generate synthetic prompt/continuation pairs
np.random.seed(42)
test_prompts = [
    "The result is ", "The URL is https://", "Output: $",
    '{"key": "', "def fn_", "import ", "The 20",
    "SELECT * FROM ", "class My", "#include <",
    "The price was $", "Call 1-800-", "Version 3.",
    "email: user@", "The hex value 0x", "Path: /usr/",
    "ratio: 3.", "Score: ", "Temperature: -", "Weight: 0.",
]
test_continuations = [
    "42", "example.com", "100",
    'value"}', "name(self):", "numpy as np", "24",
    "users", "Class:", "stdio.h>",
    "50.00", "555-0100", "14",
    "example.com", "FF", "local/bin",
    "14", "95", "40", "75",
]

scan = scan_for_boundary_artifacts(test_prompts, test_continuations, enc_cl100k)
print(f"Boundary artifact scan results:")
print(f"  Total pairs tested: {scan['mismatch'] + scan['match']}")
print(f"  Mismatches: {scan['mismatch']} ({scan['mismatch_rate']:.1%})")
print(f"  Total token difference: {scan['total_token_diff']}")
print(f"\nMismatch examples:")
for prompt, cont, diff in scan["examples"][:10]:
    print(f"  {repr(prompt + cont):<45} (token diff: {diff:+d})")

Token healing is particularly important for **structured output generation**. When a model is constrained to produce valid JSON, XML, or code, every character matters. A token boundary artifact can cause the model to assign probability mass to tokens that produce invalid syntax, because the conditioning context has been subtly corrupted by the artificial boundary.

Libraries like `guidance` and `outlines` implement token healing as part of their constrained generation pipelines. The general principle is: whenever you concatenate separately-tokenized strings and feed them to a model, you should check whether re-tokenization of the full string would produce a different sequence. If so, token healing is needed.

---

## 6. Tokenizer-Aware Prompt Optimization

In production systems, token count directly determines cost and latency. Every unnecessary token adds to the inference bill and increases time-to-first-token. Tokenizer-aware prompt optimization is the practice of rewriting prompts to minimize token count while preserving semantic content and intent.

This goes beyond simple text compression. A naive approach might shorten words or remove punctuation, but this can change how the tokenizer splits the text, sometimes increasing token count. The key insight is that optimization must be **token-boundary-aware**: you need to understand which character sequences map to single tokens and which force multi-token splits.

Practical techniques include: replacing multi-token phrases with single-token synonyms, removing redundant whitespace and formatting, using abbreviations that happen to be single tokens, and restructuring instructions to avoid repetitive token patterns. At scale, even a 10% reduction in average prompt length can save significant infrastructure costs.

In [ ]:
class TokenAwarePromptOptimizer:
    """Optimize prompts to minimize token count while preserving meaning."""
    
    def __init__(self, encoding):
        self.enc = encoding
    
    def token_count(self, text: str) -> int:
        return len(self.enc.encode(text))
    
    def token_density(self, text: str) -> float:
        """Characters per token — higher is better."""
        n_tokens = self.token_count(text)
        return len(text) / n_tokens if n_tokens > 0 else 0
    
    def find_expensive_phrases(self, text: str, window: int = 5) -> List[Tuple[str, int, float]]:
        """Find phrases in the text that have poor token density.
        
        Scans with a sliding window over words to find multi-word phrases
        that use disproportionately many tokens.
        """
        words = text.split()
        expensive = []
        
        for w in range(1, window + 1):
            for i in range(len(words) - w + 1):
                phrase = " ".join(words[i:i+w])
                n_tokens = self.token_count(phrase)
                density = len(phrase) / n_tokens if n_tokens > 0 else 0
                if density < 3.0 and n_tokens > w:  # poor density
                    expensive.append((phrase, n_tokens, density))
        
        # Deduplicate and sort by tokens (most expensive first)
        seen = set()
        unique = []
        for phrase, n_tok, density in sorted(expensive, key=lambda x: -x[1]):
            if phrase not in seen:
                seen.add(phrase)
                unique.append((phrase, n_tok, density))
        return unique[:20]
    
    def remove_redundant_whitespace(self, text: str) -> str:
        """Collapse multiple spaces, remove trailing spaces on lines."""
        text = re.sub(r" +", " ", text)
        text = re.sub(r" *\n *", "\n", text)
        return text.strip()
    
    def optimize(self, text: str) -> Tuple[str, dict]:
        """Apply basic optimizations and report savings."""
        original_tokens = self.token_count(text)
        original_chars = len(text)
        
        # Step 1: whitespace normalization
        optimized = self.remove_redundant_whitespace(text)
        
        new_tokens = self.token_count(optimized)
        
        return optimized, {
            "original_tokens": original_tokens,
            "optimized_tokens": new_tokens,
            "tokens_saved": original_tokens - new_tokens,
            "reduction_pct": (1 - new_tokens / original_tokens) * 100 if original_tokens > 0 else 0,
            "original_chars": original_chars,
            "optimized_chars": len(optimized),
        }


optimizer = TokenAwarePromptOptimizer(enc_cl100k)
print("TokenAwarePromptOptimizer ready.")

In [ ]:
# Analyze a real-world system prompt
system_prompt = """You are a helpful  AI assistant  that specializes in  answering questions 
about  machine learning  and  artificial intelligence.  You should always provide  detailed, 
accurate  responses.  When you  don't know  the answer,  you should  say so  honestly.

Please follow  these guidelines:
1.  Always cite  your sources  when possible.
2.  Use clear  and concise  language.
3.  Break down  complex topics  into  simpler  explanations.
4.  Provide  code examples  when  relevant.
5.  Be respectful  and  professional  at  all  times."""

optimized, stats = optimizer.optimize(system_prompt)

print("Optimization results:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.1f}")
    else:
        print(f"  {key}: {value}")

print(f"\nOriginal density:  {optimizer.token_density(system_prompt):.2f} chars/token")
print(f"Optimized density: {optimizer.token_density(optimized):.2f} chars/token")

In [ ]:
# Find expensive phrases in the prompt
expensive = optimizer.find_expensive_phrases(system_prompt)

print("Most token-expensive phrases in the system prompt:\n")
print(f"{'Phrase':<45} {'Tokens':>7} {'Density':>8}")
print("-" * 62)
for phrase, n_tokens, density in expensive[:15]:
    print(f"{phrase:<45} {n_tokens:>7} {density:>8.2f}")

In [ ]:
# Production cost savings calculator
def production_savings_report(avg_prompt_tokens: int, avg_output_tokens: int,
                               requests_per_day: int, reduction_pct: float,
                               cost_per_1k_input: float = 0.01,
                               cost_per_1k_output: float = 0.03):
    """Calculate production cost savings from prompt optimization."""
    # Before optimization
    daily_input_cost = (avg_prompt_tokens * requests_per_day / 1000) * cost_per_1k_input
    daily_output_cost = (avg_output_tokens * requests_per_day / 1000) * cost_per_1k_output
    daily_total = daily_input_cost + daily_output_cost
    
    # After optimization (only input tokens are reduced)
    optimized_prompt = avg_prompt_tokens * (1 - reduction_pct / 100)
    opt_daily_input = (optimized_prompt * requests_per_day / 1000) * cost_per_1k_input
    opt_daily_total = opt_daily_input + daily_output_cost
    
    daily_savings = daily_total - opt_daily_total
    
    return {
        "daily_cost_before": daily_total,
        "daily_cost_after": opt_daily_total,
        "daily_savings": daily_savings,
        "monthly_savings": daily_savings * 30,
        "annual_savings": daily_savings * 365,
        "tokens_saved_daily": (avg_prompt_tokens - optimized_prompt) * requests_per_day,
    }


# Scenario analysis
scenarios = [
    {"name": "Small SaaS",     "prompts": 500, "output": 200, "rpd": 10_000,  "reduction": 10},
    {"name": "Mid-size API",   "prompts": 800, "output": 400, "rpd": 100_000, "reduction": 10},
    {"name": "Enterprise",     "prompts": 1200, "output": 600, "rpd": 1_000_000, "reduction": 10},
    {"name": "Enterprise+15%", "prompts": 1200, "output": 600, "rpd": 1_000_000, "reduction": 15},
]

print(f"{'Scenario':<18} {'Daily Cost':>12} {'After Opt':>12} {'Monthly Save':>14} {'Annual Save':>14}")
print("-" * 74)
for s in scenarios:
    report = production_savings_report(s["prompts"], s["output"], s["rpd"], s["reduction"])
    print(f"{s['name']:<18} ${report['daily_cost_before']:>10,.2f} ${report['daily_cost_after']:>10,.2f} "
          f"${report['monthly_savings']:>12,.2f} ${report['annual_savings']:>12,.2f}")

In [ ]:
# Visualize savings across different optimization levels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

reduction_levels = np.arange(0, 26, 1)
rpd_levels = [10_000, 100_000, 1_000_000]
rpd_labels = ["10K req/day", "100K req/day", "1M req/day"]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

for rpd, label, color in zip(rpd_levels, rpd_labels, colors):
    annual_savings = []
    for red in reduction_levels:
        r = production_savings_report(800, 400, rpd, red)
        annual_savings.append(r["annual_savings"])
    axes[0].plot(reduction_levels, annual_savings, label=label, color=color, linewidth=2)

axes[0].set_xlabel("Token Reduction (%)")
axes[0].set_ylabel("Annual Savings ($)")
axes[0].set_title("Annual Savings by Optimization Level")
axes[0].legend()
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Token density comparison across tokenizers
sample_texts = list(corpus.values())
densities = {}
for tok_name, encode_fn in TOKENIZERS.items():
    text_densities = []
    for text in sample_texts:
        n_tokens = len(encode_fn(text))
        text_densities.append(len(text) / n_tokens if n_tokens > 0 else 0)
    densities[tok_name] = text_densities

x = np.arange(len(corpus))
width = 0.2
for i, (tok_name, dens_values) in enumerate(densities.items()):
    axes[1].bar(x + i * width, dens_values, width, label=tok_name)

axes[1].set_xlabel("Corpus")
axes[1].set_ylabel("Characters per Token")
axes[1].set_title("Token Density by Corpus and Tokenizer")
axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(list(corpus.keys()), rotation=45, ha="right", fontsize=7)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

The savings analysis above shows that tokenizer-aware optimization is not just a micro-optimization. For high-volume production systems, even modest token reductions translate to tens or hundreds of thousands of dollars in annual savings. The compounding effect of daily savings over a year is substantial.

Token density (characters per token) is the inverse of character fertility and provides an intuitive measure of tokenizer efficiency. Higher density means each token carries more information, which translates directly to lower costs and better utilization of the context window. The variation across languages reinforces the earlier finding about tokenizer equity — a system optimized for English may be significantly less cost-effective for other languages.

---

## Building a Tokenizer Audit Toolkit

Let us combine the tools we have built into a single audit function that can comprehensively evaluate any model's tokenizer. This toolkit produces a report covering all the dimensions we have explored: vocabulary statistics, fertility analysis, compression ratios, boundary artifact susceptibility, and cost projections.

A tokenizer audit is valuable when selecting a base model for a new project, when evaluating whether tokenizer extension is needed for a target domain, or when debugging unexpected model behavior that might stem from tokenization artifacts.

In [ ]:
def audit_tokenizer(name: str, encode_fn: callable, vocab_size: int,
                     test_corpus: Dict[str, str],
                     boundary_tests: Optional[List[Tuple[str, str]]] = None) -> dict:
    """Run a comprehensive audit on a tokenizer."""
    report = {"name": name, "vocab_size": vocab_size}
    
    # 1. Fertility analysis
    fertility_results = {}
    for corpus_name, text in test_corpus.items():
        n_tokens = len(encode_fn(text))
        n_words = len(text.split())
        n_chars = len(text)
        fertility_results[corpus_name] = {
            "word_fertility": n_tokens / max(1, n_words),
            "char_fertility": n_tokens / max(1, n_chars),
            "n_tokens": n_tokens,
        }
    report["fertility"] = fertility_results
    
    # 2. Average metrics
    avg_word_fert = np.mean([v["word_fertility"] for v in fertility_results.values()])
    avg_char_fert = np.mean([v["char_fertility"] for v in fertility_results.values()])
    report["avg_word_fertility"] = avg_word_fert
    report["avg_char_fertility"] = avg_char_fert
    
    # 3. Compression ratio
    all_text = " ".join(test_corpus.values())
    all_tokens = len(encode_fn(all_text))
    utf8_bytes = len(all_text.encode("utf-8"))
    bits_per_token = math.ceil(math.log2(vocab_size))
    encoded_bytes = (all_tokens * bits_per_token) / 8
    report["compression_ratio"] = utf8_bytes / encoded_bytes if encoded_bytes > 0 else 0
    report["bpc"] = (all_tokens * math.log2(vocab_size)) / max(1, len(all_text))
    
    # 4. Embedding cost
    d_model = 4096
    report["embedding_params_M"] = vocab_size * d_model / 1e6
    report["embedding_memory_fp16_MB"] = vocab_size * d_model * 2 / (1024**2)
    
    return report


# Run audit for all tokenizers
print("=" * 70)
print("TOKENIZER AUDIT REPORT")
print("=" * 70)

for tok_name, encode_fn in TOKENIZERS.items():
    report = audit_tokenizer(tok_name, encode_fn, VOCAB_SIZES[tok_name], corpus)
    print(f"\n--- {report['name']} ---")
    print(f"  Vocabulary size:       {report['vocab_size']:>10,}")
    print(f"  Avg word fertility:    {report['avg_word_fertility']:>10.3f}")
    print(f"  Avg char fertility:    {report['avg_char_fertility']:>10.4f}")
    print(f"  Compression ratio:     {report['compression_ratio']:>10.2f}x")
    print(f"  Bits per character:    {report['bpc']:>10.3f}")
    print(f"  Embedding params:      {report['embedding_params_M']:>10.1f}M")
    print(f"  Embedding mem (FP16):  {report['embedding_memory_fp16_MB']:>10.1f} MB")
    print(f"  Fertility by corpus:")
    for c, v in report["fertility"].items():
        print(f"    {c:<22} {v['word_fertility']:.2f} tok/word  ({v['n_tokens']} tokens)")

In [ ]:
# Summary radar chart comparing tokenizers
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

metrics = ["Vocab Size\n(normalized)", "Avg Fertility\n(inverted)", 
           "Compression\nRatio", "Embedding\nEfficiency", "CJK\nFertility (inv)"]
n_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

colors_radar = ["#2196F3", "#FF9800", "#4CAF50", "#E91E63"]

for (tok_name, encode_fn), color in zip(TOKENIZERS.items(), colors_radar):
    report = audit_tokenizer(tok_name, encode_fn, VOCAB_SIZES[tok_name], corpus)
    
    # Normalize metrics to 0-1 scale
    vocab_norm = min(VOCAB_SIZES[tok_name] / 200000, 1.0)
    fert_inv = 1.0 / report["avg_word_fertility"]  # lower fertility = better
    comp_norm = min(report["compression_ratio"] / 5.0, 1.0)
    emb_eff = 1.0 - min(report["embedding_params_M"] / 1000, 1.0)  # fewer params = better
    
    # CJK fertility (use Chinese + Japanese average)
    cjk_fert = np.mean([
        report["fertility"].get("Chinese", {}).get("word_fertility", 3.0),
        report["fertility"].get("Japanese", {}).get("word_fertility", 3.0),
    ])
    cjk_inv = 1.0 / cjk_fert
    
    values = [vocab_norm, fert_inv, comp_norm, emb_eff, cjk_inv]
    values += values[:1]  # close polygon
    
    ax.plot(angles, values, 'o-', linewidth=2, label=tok_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_ylim(0, 1)
ax.set_title("Tokenizer Comparison Radar Chart", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

plt.tight_layout()
plt.show()

---

## Summary

In this expert-level notebook, we explored the deep interactions between tokenization and model behavior:

**Tokenizer-Model Interactions:** Vocabulary size directly determines embedding parameter count and memory footprint. Token count determines inference FLOPs. The choice of tokenizer is an architectural decision with cascading effects on cost, performance, and multilingual capability.

**Fertility Analysis:** We built a fertility analyzer and demonstrated that fertility varies dramatically across languages and domains. This variation maps directly to economic cost — underrepresented languages pay a token tax that can be 2-3x that of English.

**Compression and Information Theory:** We measured bits-per-character and compression ratios, showing that learned tokenizers are better than naive encoding but still far from the information-theoretic optimum. The gap represents the cost of the BPE greedy heuristic.

**Tokenizer Extension and Merging:** We built a `TokenizerExtender` that identifies high-fertility domain terms and adds them to an existing tokenizer, with proper embedding initialization. We also explored vocabulary merging between models.

**Token Healing:** We demonstrated the token boundary artifact problem and implemented a `TokenHealer` that resolves mismatches between separate and joint tokenization, critical for structured output generation.

**Prompt Optimization:** We built a `TokenAwarePromptOptimizer` and showed that even modest token reductions translate to substantial cost savings at production scale.

**Next:** Continue to `build.ipynb` where you will apply these tools to build a complete tokenizer audit pipeline for a real model deployment.